In [1]:
!uv add pystac-client planetary-computer rasterio geopandas shapely pyproj pandas numpy requests tqdm

⠹ jsonschema-specifications==2025.9.1                                           Resolved 72 packages in 352ms
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/660.75 KiB          
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/660.75 KiB        
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/660.75 KiB        
⠙ Preparing packages... (0/1)------------------- 48.00 KiB/660.75 KiB        
⠙ Preparing packages... (0/1)------------------- 62.39 KiB/660.75 KiB        
⠙ Preparing packages... (0/1)------------------- 78.39 KiB/660.75 KiB        
⠙ Preparing packages... (0/1)------------------- 94.39 KiB/660.75 KiB        
⠙ Preparing packages... (0/1)------------------- 110.39 KiB/660.75 KiB       
⠙ Preparing packages... (0/1)------------------- 126.39 KiB/660.75 KiB       
⠙ Preparing packages... (0/1)------------------- 142.39 KiB/660.75 KiB       
⠙ Preparing packages... (0/1)

In [2]:
# ============================================================
# HLS S30 + L30 COUNTY-SCALE VI SUMMARY
# Example: Delaware soybean pixels using local CDL soybean mask
#
# Output:
#   county-level monthly + seasonal VI summary CSV
#
# Uses:
#   Microsoft Planetary Computer HLS:
#     hls2-s30 = Sentinel-2-derived HLS
#     hls2-l30 = Landsat-derived HLS
#
# Notes:
#   - This is for county summary, not pixel export.
#   - You provide the CDL soybean mask GeoTIFF.
#   - Works best for 2020+ through Planetary Computer HLS.
# ============================================================

import calendar
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from tqdm import tqdm

import rasterio
from rasterio.windows import from_bounds
from rasterio.features import rasterize
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT

from shapely.geometry import mapping

from pystac_client import Client
import planetary_computer as pc


# ============================================================
# 1. USER SETTINGS
# ============================================================

STATE_FIPS = "10"          # Delaware = 10
STATE_NAME = "Delaware"
YEAR = 2022

# Optional: set to "Kent", "Sussex", or "New Castle" for one county only.
# Set to None to process all Delaware counties.
COUNTY_NAME_FILTER = None

# Growing season months for soybean.
MONTHS = [4,5, 6, 7, 8, 9,10]

# Path to your CDL soybean mask.
# If this is already a binary soybean mask, use CDL_MASK_MODE = "binary".
# If this is raw CDL, use CDL_MASK_MODE = "cdl_code" and SOYBEAN_CDL_CODE = 5.
CDL_PATH = Path("/Users/samarranjit/Library/CloudStorage/OneDrive-TexasStateUniversity/ChoLab/USDA Crop yield Stability Study/EO based yield prediction/data_preparation/data/cdl_masks/cdl_soybeans_DE_2022.tif")

CDL_MASK_MODE = "binary"   # "binary" or "cdl_code"
SOYBEAN_CDL_CODE = 5       # CDL soybean class code, used only if mode is "cdl_code"

OUT_DIR = Path("/Users/samarranjit/Library/CloudStorage/OneDrive-TexasStateUniversity/ChoLab/USDA Crop yield Stability Study/EO based yield prediction/data_preparation/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = OUT_DIR / f"{STATE_NAME.lower()}_soybean_hls_vi_summary_{YEAR}.csv"

# HLS search settings
MAX_SCENE_CLOUD = 80       # scene-level filter; pixel-level Fmask still does real masking
MAX_ITEMS_PER_MONTH = 500

# HLS Fmask masking.
# Bit 1: cloud
# Bit 2: adjacent to cloud/shadow
# Bit 3: cloud shadow
# Bit 4: snow/ice
# Bit 5: water
#
# If you want more valid pixels, you may remove bit 2 from BAD_FMASK_BITS.
BAD_FMASK_BITS = [1, 2, 3, 4, 5]

MASK_HIGH_AEROSOL = True   # bits 6-7; masks aerosol level 3

REFLECTANCE_SCALE = 0.0001
EPS = 1e-6

VI_NAMES = ["NDVI", "GCVI", "EVI", "NDWI"]


# ============================================================
# 2. HLS BAND MAPPING
# ============================================================

# S30 Sentinel-derived HLS bands
S30_BANDS = {
    "blue": "B02",
    "green": "B03",
    "red": "B04",
    "nir": "B8A",      # narrow NIR
    "swir1": "B11",
    "swir2": "B12",
    "fmask": "Fmask",
}

# L30 Landsat-derived HLS bands
L30_BANDS = {
    "blue": "B02",
    "green": "B03",
    "red": "B04",
    "nir": "B05",      # narrow NIR
    "swir1": "B06",
    "swir2": "B07",
    "fmask": "Fmask",
}


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def download_counties_for_state(state_fips: str, tiger_year: int = 2023) -> gpd.GeoDataFrame:
    """
    Download Census generalized county boundaries and filter to one state.
    """
    url = (
        f"https://www2.census.gov/geo/tiger/GENZ{tiger_year}/shp/"
        f"cb_{tiger_year}_us_county_500k.zip"
    )

    with tempfile.TemporaryDirectory() as tmpdir:
        zip_path = Path(tmpdir) / "counties.zip"

        print(f"Downloading county boundaries from Census TIGER/Line {tiger_year}...")
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        zip_path.write_bytes(r.content)

        counties = gpd.read_file(zip_path)

    counties = counties[counties["STATEFP"] == state_fips].copy()
    counties = counties.to_crs("EPSG:4326")

    if COUNTY_NAME_FILTER is not None:
        counties = counties[
            counties["NAME"].str.lower().str.contains(COUNTY_NAME_FILTER.lower())
        ].copy()

    if counties.empty:
        raise ValueError("No counties found. Check STATE_FIPS or COUNTY_NAME_FILTER.")

    counties["county_idx"] = np.arange(1, len(counties) + 1, dtype=np.int16)

    print("Counties selected:")
    print(counties[["GEOID", "NAME"]].to_string(index=False))

    return counties


def open_cdl_target_grid(cdl_path: Path, counties_4326: gpd.GeoDataFrame):
    """
    Read the CDL only for the county/state bounding area.
    The CDL grid becomes the target grid for all HLS reprojection.
    """
    if not cdl_path.exists():
        raise FileNotFoundError(f"CDL file not found: {cdl_path}")

    with rasterio.open(cdl_path) as cdl:
        cdl_crs = cdl.crs
        if cdl_crs is None:
            raise ValueError("CDL raster has no CRS.")

        counties_cdl = counties_4326.to_crs(cdl_crs)
        geom_union = counties_cdl.geometry.unary_union

        bounds = geom_union.bounds
        window = from_bounds(*bounds, transform=cdl.transform)
        window = window.round_offsets().round_lengths()

        cdl_arr = cdl.read(
            1,
            window=window,
            boundless=True,
            fill_value=0
        )

        target_transform = cdl.window_transform(window)
        target_crs = cdl.crs
        target_height, target_width = cdl_arr.shape

    if CDL_MASK_MODE == "binary":
        crop_mask = cdl_arr > 0
    elif CDL_MASK_MODE == "cdl_code":
        crop_mask = cdl_arr == SOYBEAN_CDL_CODE
    else:
        raise ValueError("CDL_MASK_MODE must be 'binary' or 'cdl_code'.")

    print(f"Target grid shape: {target_height} rows × {target_width} cols")
    print(f"Crop pixels in target grid: {int(crop_mask.sum()):,}")

    if crop_mask.sum() == 0:
        raise ValueError("No soybean pixels found in the CDL mask area.")

    return crop_mask, target_transform, target_crs, target_height, target_width


def rasterize_counties(counties_4326, target_crs, target_transform, target_shape):
    """
    Rasterize county polygons to the CDL/HLS target grid.
    """
    counties_target = counties_4326.to_crs(target_crs)

    shapes = [
        (geom, int(idx))
        for geom, idx in zip(counties_target.geometry, counties_target["county_idx"])
        if geom is not None and not geom.is_empty
    ]

    county_id = rasterize(
        shapes=shapes,
        out_shape=target_shape,
        transform=target_transform,
        fill=0,
        dtype="int16",
        all_touched=False,
    )

    return county_id


def connect_planetary_computer():
    """
    Open Planetary Computer STAC. pc.sign_inplace signs asset URLs automatically.
    """
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    return catalog


def month_date_range(year: int, month: int):
    last_day = calendar.monthrange(year, month)[1]
    return f"{year}-{month:02d}-01/{year}-{month:02d}-{last_day:02d}"


def cloud_cover(item):
    """
    Get scene-level cloud cover safely.
    """
    val = item.properties.get("eo:cloud_cover")
    if val is None:
        val = item.properties.get("cloud_cover")
    if val is None:
        return 999.0
    return float(val)


def search_hls_items(catalog, counties_4326, year: int, month: int):
    """
    Search both HLS S30 and L30 over the county/state geometry.
    """
    geom = mapping(counties_4326.geometry.unary_union)
    dt = month_date_range(year, month)

    search = catalog.search(
        collections=["hls2-s30", "hls2-l30"],
        intersects=geom,
        datetime=dt,
        max_items=MAX_ITEMS_PER_MONTH,
    )

    items = list(search.items())

    # Filter by scene-level cloud cover after search.
    items = [
        item for item in items
        if cloud_cover(item) <= MAX_SCENE_CLOUD
    ]

    items = sorted(items, key=lambda x: cloud_cover(x))

    return items


def get_band_map(item):
    """
    Return correct band names for S30 or L30 item.
    """
    if item.collection_id == "hls2-s30":
        return S30_BANDS
    elif item.collection_id == "hls2-l30":
        return L30_BANDS
    else:
        raise ValueError(f"Unknown HLS collection: {item.collection_id}")


def read_asset_to_target_grid(
    asset_href,
    target_crs,
    target_transform,
    target_height,
    target_width,
    resampling,
    dst_dtype="float32",
    dst_nodata=None,
):
    """
    Read one COG asset and warp it directly to the CDL target grid.

    This avoids downloading the full HLS tile.
    Rasterio/GDAL reads only the required COG windows internally.
    """
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
        with rasterio.open(asset_href) as src:
            src_nodata = src.nodata

            with WarpedVRT(
                src,
                crs=target_crs,
                transform=target_transform,
                width=target_width,
                height=target_height,
                resampling=resampling,
                src_nodata=src_nodata,
                nodata=dst_nodata if dst_nodata is not None else src_nodata,
            ) as vrt:
                arr = vrt.read(1).astype(dst_dtype)

    return arr


def hls_fmask_valid(fmask):
    """
    Convert HLS Fmask to valid clear-pixel mask.
    """
    fmask = fmask.astype("uint8")

    bad = np.zeros(fmask.shape, dtype=bool)

    for bit in BAD_FMASK_BITS:
        bad |= ((fmask >> bit) & 1).astype(bool)

    if MASK_HIGH_AEROSOL:
        aerosol = (fmask >> 6) & 3
        bad |= aerosol == 3

    return ~bad


def scale_reflectance(arr):
    """
    Convert HLS integer reflectance to scaled reflectance.
    """
    arr = arr.astype("float32")

    # Common HLS fill value is -9999.
    arr[arr <= -9990] = np.nan

    arr *= REFLECTANCE_SCALE

    # Conservative physical sanity range.
    arr[(arr < -0.2) | (arr > 1.6)] = np.nan

    return arr


def compute_vis(blue, green, red, nir, swir1):
    """
    Compute vegetation indices.
    NDWI here is NIR-SWIR1 moisture-style NDWI, not green-NIR water NDWI.
    """
    vis = {}

    ndvi_den = nir + red
    vis["NDVI"] = np.where(
        np.abs(ndvi_den) > EPS,
        (nir - red) / ndvi_den,
        np.nan,
    )

    vis["GCVI"] = np.where(
        green > EPS,
        (nir / green) - 1.0,
        np.nan,
    )

    evi_den = nir + 6.0 * red - 7.5 * blue + 1.0
    vis["EVI"] = np.where(
        np.abs(evi_den) > EPS,
        2.5 * (nir - red) / evi_den,
        np.nan,
    )

    ndwi_den = nir + swir1
    vis["NDWI"] = np.where(
        np.abs(ndwi_den) > EPS,
        (nir - swir1) / ndwi_den,
        np.nan,
    )

    for k in vis:
        vis[k] = vis[k].astype("float32")

    return vis


def empty_county_records(counties_4326, crop_mask, county_id):
    """
    Initialize output row for each county.
    """
    records = {}

    for _, r in counties_4326.iterrows():
        idx = int(r["county_idx"])
        county_crop_mask = crop_mask & (county_id == idx)
        crop_pixels = int(county_crop_mask.sum())

        records[idx] = {
            "state_fips": r["STATEFP"],
            "county_fips": r["COUNTYFP"],
            "geoid": r["GEOID"],
            "county_name": r["NAME"],
            "year": YEAR,
            "crop": "soybeans",
            "crop_pixel_count": crop_pixels,
        }

    return records


def summarize_array_by_county(
    arr,
    feature_prefix,
    records,
    crop_mask,
    county_id,
    counties_4326,
):
    """
    Add county-level summary stats for one raster array.
    """
    for _, r in counties_4326.iterrows():
        idx = int(r["county_idx"])

        county_crop = crop_mask & (county_id == idx)
        crop_n = int(county_crop.sum())

        if crop_n == 0:
            values = np.array([], dtype="float32")
        else:
            values = arr[county_crop]
            values = values[np.isfinite(values)]

        valid_n = int(values.size)

        if valid_n == 0:
            records[idx][f"{feature_prefix}_mean"] = np.nan
            records[idx][f"{feature_prefix}_median"] = np.nan
            records[idx][f"{feature_prefix}_std"] = np.nan
            records[idx][f"{feature_prefix}_min"] = np.nan
            records[idx][f"{feature_prefix}_max"] = np.nan
            records[idx][f"{feature_prefix}_valid_pixel_count"] = 0
            records[idx][f"{feature_prefix}_valid_fraction"] = 0.0
        else:
            records[idx][f"{feature_prefix}_mean"] = float(np.nanmean(values))
            records[idx][f"{feature_prefix}_median"] = float(np.nanmedian(values))
            records[idx][f"{feature_prefix}_std"] = float(np.nanstd(values))
            records[idx][f"{feature_prefix}_min"] = float(np.nanmin(values))
            records[idx][f"{feature_prefix}_max"] = float(np.nanmax(values))
            records[idx][f"{feature_prefix}_valid_pixel_count"] = valid_n
            records[idx][f"{feature_prefix}_valid_fraction"] = valid_n / crop_n if crop_n > 0 else np.nan


def process_one_hls_item(
    item,
    target_crs,
    target_transform,
    target_height,
    target_width,
):
    """
    Read one HLS item, cloud-mask it, compute VI rasters on the target grid.
    """
    band_map = get_band_map(item)

    # Read Fmask first.
    fmask = read_asset_to_target_grid(
        item.assets[band_map["fmask"]].href,
        target_crs,
        target_transform,
        target_height,
        target_width,
        resampling=Resampling.nearest,
        dst_dtype="uint8",
        dst_nodata=255,
    )

    clear = hls_fmask_valid(fmask)

    # Read reflectance bands.
    blue = scale_reflectance(
        read_asset_to_target_grid(
            item.assets[band_map["blue"]].href,
            target_crs,
            target_transform,
            target_height,
            target_width,
            resampling=Resampling.bilinear,
        )
    )

    green = scale_reflectance(
        read_asset_to_target_grid(
            item.assets[band_map["green"]].href,
            target_crs,
            target_transform,
            target_height,
            target_width,
            resampling=Resampling.bilinear,
        )
    )

    red = scale_reflectance(
        read_asset_to_target_grid(
            item.assets[band_map["red"]].href,
            target_crs,
            target_transform,
            target_height,
            target_width,
            resampling=Resampling.bilinear,
        )
    )

    nir = scale_reflectance(
        read_asset_to_target_grid(
            item.assets[band_map["nir"]].href,
            target_crs,
            target_transform,
            target_height,
            target_width,
            resampling=Resampling.bilinear,
        )
    )

    swir1 = scale_reflectance(
        read_asset_to_target_grid(
            item.assets[band_map["swir1"]].href,
            target_crs,
            target_transform,
            target_height,
            target_width,
            resampling=Resampling.bilinear,
        )
    )

    valid_reflectance = (
        clear
        & np.isfinite(blue)
        & np.isfinite(green)
        & np.isfinite(red)
        & np.isfinite(nir)
        & np.isfinite(swir1)
    )

    vis = compute_vis(blue, green, red, nir, swir1)

    for name in vis:
        vis[name][~valid_reflectance] = np.nan

    return vis


def nanmedian_stack(arrays):
    """
    Robust nanmedian stack.
    """
    if len(arrays) == 0:
        return None

    stack = np.stack(arrays, axis=0).astype("float32")

    with np.errstate(all="ignore"):
        comp = np.nanmedian(stack, axis=0).astype("float32")

    return comp


# ============================================================
# 4. MAIN PROCESS
# ============================================================

def main():
    print("\n============================================================")
    print("HLS S30 + L30 county-scale VI summary")
    print(f"State: {STATE_NAME}")
    print(f"Year : {YEAR}")
    print("============================================================\n")

    # --------------------------------------------------------
    # County boundaries
    # --------------------------------------------------------
    counties = download_counties_for_state(STATE_FIPS)

    # --------------------------------------------------------
    # CDL target grid + crop mask
    # --------------------------------------------------------
    crop_mask, target_transform, target_crs, target_height, target_width = (
        open_cdl_target_grid(CDL_PATH, counties)
    )

    target_shape = (target_height, target_width)

    county_id = rasterize_counties(
        counties,
        target_crs,
        target_transform,
        target_shape,
    )

    records = empty_county_records(counties, crop_mask, county_id)

    # --------------------------------------------------------
    # Connect to Planetary Computer
    # --------------------------------------------------------
    catalog = connect_planetary_computer()

    # Store monthly composites for seasonal summaries.
    seasonal_composites = {vi: [] for vi in VI_NAMES}

    # --------------------------------------------------------
    # Monthly loop
    # --------------------------------------------------------
    for month in MONTHS:
        print(f"\n---------------- Month {month:02d} ----------------")

        items = search_hls_items(catalog, counties, YEAR, month)
        print(f"Found {len(items)} HLS items after scene-cloud filtering.")

        for idx in records:
            records[idx][f"month_{month:02d}_hls_item_count"] = len(items)

        monthly_vi_arrays = {vi: [] for vi in VI_NAMES}
        used_items = 0

        for item in tqdm(items, desc=f"Processing HLS items {YEAR}-{month:02d}"):
            try:
                vis = process_one_hls_item(
                    item,
                    target_crs,
                    target_transform,
                    target_height,
                    target_width,
                )

                # Only store VI arrays that have at least some valid crop pixels.
                any_valid = False
                for vi_name, arr in vis.items():
                    crop_values = arr[crop_mask & (county_id > 0)]
                    if np.isfinite(crop_values).any():
                        monthly_vi_arrays[vi_name].append(arr)
                        any_valid = True

                if any_valid:
                    used_items += 1

            except Exception as e:
                print(f"WARNING: skipping item {item.id}: {e}")
                continue

        print(f"Used {used_items} HLS items with valid crop pixels.")

        for idx in records:
            records[idx][f"month_{month:02d}_hls_item_used_count"] = used_items

        # Monthly median composite per VI, then county summary.
        for vi_name in VI_NAMES:
            comp = nanmedian_stack(monthly_vi_arrays[vi_name])

            if comp is None:
                print(f"  {vi_name}: no valid composite.")
                continue

            seasonal_composites[vi_name].append(comp)

            feature_prefix = f"{vi_name}_month_{month:02d}"
            summarize_array_by_county(
                comp,
                feature_prefix,
                records,
                crop_mask,
                county_id,
                counties,
            )

            print(f"  {vi_name}: monthly composite summarized.")

    # --------------------------------------------------------
    # Seasonal summaries
    # --------------------------------------------------------
    print("\n---------------- Seasonal summaries ----------------")

    for vi_name in VI_NAMES:
        comps = seasonal_composites[vi_name]

        if len(comps) == 0:
            print(f"{vi_name}: no seasonal composites.")
            continue

        stack = np.stack(comps, axis=0).astype("float32")

        with np.errstate(all="ignore"):
            season_mean = np.nanmean(stack, axis=0).astype("float32")
            season_max = np.nanmax(stack, axis=0).astype("float32")
            season_min = np.nanmin(stack, axis=0).astype("float32")
            season_range = (season_max - season_min).astype("float32")

        summarize_array_by_county(
            season_mean,
            f"{vi_name}_season_mean",
            records,
            crop_mask,
            county_id,
            counties,
        )

        summarize_array_by_county(
            season_max,
            f"{vi_name}_season_max",
            records,
            crop_mask,
            county_id,
            counties,
        )

        summarize_array_by_county(
            season_range,
            f"{vi_name}_season_range",
            records,
            crop_mask,
            county_id,
            counties,
        )

        print(f"{vi_name}: seasonal mean/max/range summarized.")

    # --------------------------------------------------------
    # Save output
    # --------------------------------------------------------
    df = pd.DataFrame.from_dict(records, orient="index")
    df = df.sort_values(["geoid"]).reset_index(drop=True)

    df.to_csv(OUT_CSV, index=False)

    print("\n============================================================")
    print("Done.")
    print(f"Saved: {OUT_CSV}")
    print("Preview:")
    print(df.head().to_string(index=False))
    print("============================================================")


if __name__ == "__main__":
    main()


HLS S30 + L30 county-scale VI summary
State: Delaware
Year : 2022

Counties selected:
GEOID       NAME
10001       Kent
10003 New Castle
10005     Sussex
Target grid shape: 5131 rows × 3084 cols
Crop pixels in target grid: 1,728,956


/var/folders/rg/ryvm2std1qj_l_1rfp3f8_b80000gn/T/ipykernel_34944/787341238.py:174: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom_union = counties_cdl.geometry.unary_union



---------------- Month 04 ----------------


/var/folders/rg/ryvm2std1qj_l_1rfp3f8_b80000gn/T/ipykernel_34944/787341238.py:263: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = mapping(counties_4326.geometry.unary_union)


Found 29 HLS items after scene-cloud filtering.


...

/var/folders/rg/ryvm2std1qj_l_1rfp3f8_b80000gn/T/ipykernel_34944/787341238.py:584: RuntimeWarning: All-NaN slice encountered
  comp = np.nanmedian(stack, axis=0).astype("float32")


  GCVI: monthly composite summarized.


/var/folders/rg/ryvm2std1qj_l_1rfp3f8_b80000gn/T/ipykernel_34944/787341238.py:584: RuntimeWarning: All-NaN slice encountered
  comp = np.nanmedian(stack, axis=0).astype("float32")


  EVI: monthly composite summarized.


KeyboardInterrupt: 